# 12 — Lookback & Cliquet Options

**Lookback options** — payoff depends on extremum of path:
- Floating lookback: call pays $S_T - S_{\min}$, put pays $S_{\max} - S_T$
- Fixed lookback: call pays $\max(S_{\max} - K, 0)$

**Cliquet / forward-start options** — strike set at a future date:
- Forward-start call/put
- Cliquet (ratchet) with local/global caps and floors

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.lookback import floating_lookback_price, fixed_lookback_price
from ql_jax.engines.analytic.cliquet import cliquet_price, forward_start_price

S, K, T = 100.0, 100.0, 1.0
r, q, sigma = 0.05, 0.02, 0.30
S_MIN = 95.0   # running minimum
S_MAX = 110.0   # running maximum

---
## 1. Floating Lookback

In [ ]:
# call: S_T - S_min; put: S_max - S_T
fl_call = float(floating_lookback_price(S, S_MIN, S_MAX, T, r, q, sigma, option_type='call'))
fl_put  = float(floating_lookback_price(S, S_MIN, S_MAX, T, r, q, sigma, option_type='put'))

print(f"Floating lookback call (S_min={S_MIN}): {fl_call:.6f}")
print(f"Floating lookback put  (S_max={S_MAX}): {fl_put:.6f}")

---
## 2. Fixed Lookback

In [ ]:
# call: max(S_max - K, 0); put: max(K - S_min, 0)
fx_call = float(fixed_lookback_price(S, K, S_MIN, S_MAX, T, r, q, sigma, option_type='call'))
fx_put  = float(fixed_lookback_price(S, K, S_MIN, S_MAX, T, r, q, sigma, option_type='put'))

print(f"Fixed lookback call : {fx_call:.6f}")
print(f"Fixed lookback put  : {fx_put:.6f}")

---
## 3. QuantLib Comparison

In [ ]:
today = QL.Date(15, 6, 2024)
QL.Settings.instance().evaluationDate = today
maturity = QL.Date(15, 6, 2025)

dc = QL.Actual365Fixed()
cal = QL.NullCalendar()

spot_h = QL.QuoteHandle(QL.SimpleQuote(S))
r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, r, dc))
q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(today, q, dc))
vol_ts = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(today, cal, sigma, dc))
bsm    = QL.BlackScholesMertonProcess(spot_h, q_ts, r_ts, vol_ts)

# QL floating lookback call
payoff_fl = QL.FloatingTypePayoff(QL.Option.Call)
exercise = QL.EuropeanExercise(maturity)
ql_fl_call_opt = QL.ContinuousFloatingLookbackOption(S_MIN, payoff_fl, exercise)
ql_fl_call_opt.setPricingEngine(QL.AnalyticContinuousFloatingLookbackEngine(bsm))
ql_fl_call = ql_fl_call_opt.NPV()

# QL fixed lookback call
payoff_fx = QL.PlainVanillaPayoff(QL.Option.Call, K)
ql_fx_call_opt = QL.ContinuousFixedLookbackOption(S_MAX, payoff_fx, exercise)
ql_fx_call_opt.setPricingEngine(QL.AnalyticContinuousFixedLookbackEngine(bsm))
ql_fx_call = ql_fx_call_opt.NPV()

print(f"\n{'Type':<25s} {'QL':>12s} {'ql-jax':>12s} {'Diff':>12s}")
print("-" * 65)
print(f"{'Float lookback call':<25s} {ql_fl_call:>12.6f} {fl_call:>12.6f} {abs(ql_fl_call-fl_call):>12.2e}")
print(f"{'Fixed lookback call':<25s} {ql_fx_call:>12.6f} {fx_call:>12.6f} {abs(ql_fx_call-fx_call):>12.2e}")

---
## 4. Forward-Start Option

In [ ]:
# Strike will be set at T1 = fraction of spot at that time
T1, T2 = 0.25, 1.0  # strike set in 3 months, expires in 1 year

fs_call = float(forward_start_price(S, T1, T2, r, q, sigma, option_type='call', strike_ratio=1.0))
fs_put  = float(forward_start_price(S, T1, T2, r, q, sigma, option_type='put', strike_ratio=1.0))

print(f"Forward-start call (ATM, 3m→1Y) : {fs_call:.6f}")
print(f"Forward-start put                : {fs_put:.6f}")

---
## 5. Cliquet (Ratchet)

In [ ]:
# Quarterly resets over 2 years, with local floor and cap
reset_dates = jnp.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0])

cli_uncapped = float(cliquet_price(
    S, 2.0, r, q, sigma, reset_dates))

cli_capped = float(cliquet_price(
    S, 2.0, r, q, sigma, reset_dates,
    local_floor=-0.05, local_cap=0.10,
    global_floor=0.0, global_cap=0.30))

print(f"Cliquet (uncapped)               : {cli_uncapped:.6f}")
print(f"Cliquet (local ±5/10%, global 0-30%) : {cli_capped:.6f}")

---
## 6. AD Greeks

In [ ]:
import matplotlib.pyplot as plt

# Lookback delta
def fl_call_fn(spot):
    return floating_lookback_price(spot, S_MIN, S_MAX, T, r, q, sigma, option_type='call')

delta_fn = jax.grad(fl_call_fn)
spots = jnp.linspace(80, 130, 80)
deltas = [float(delta_fn(s)) for s in spots]

# Cliquet vega
def cli_fn(vol):
    return cliquet_price(S, 2.0, r, q, vol, reset_dates,
                         local_floor=-0.05, local_cap=0.10,
                         global_floor=0.0, global_cap=0.30)

vega_fn = jax.grad(cli_fn)
vols = jnp.linspace(0.10, 0.60, 60)
vegas = [float(vega_fn(v)) for v in vols]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.array(spots), deltas, 'b-', linewidth=2)
ax1.set_xlabel('Spot'); ax1.set_ylabel('Delta')
ax1.set_title('Floating Lookback Call — Delta via AD')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(vols)*100, vegas, 'r-', linewidth=2)
ax2.set_xlabel('Vol (%)')
ax2.set_ylabel('Vega')
ax2.set_title('Cliquet Vega (local caps, global cap)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 7. Exercises

1. **MC lookback**: Use `mc_lookback_price` to verify the analytic floating lookback, varying `n_steps`.
2. **Cliquet structures**: Price a reverse cliquet (negative returns benefit) by adjusting caps/floors.
3. **Partial lookback**: Use `partial_fixed_lookback_price` with a lookback window shorter than the option life.

---
*End of Notebook 12*